### 스마트서울 도시데이터 센서(S-DoT) 환경정보

In [1]:
%useLatestDescriptors
%use dataframe
%use datetime

In [2]:
val url = "/Users/unchil/Desktop/S-DoT_위치정보.xlsx"

In [3]:
val df = DataFrame.readExcel(url)
df

No,모델 시리얼(*),주소,좌표 구분코드,위도,경도,변경 전 시리얼,변경 전 시리얼(데이터 상 표기)
1.000000,V02Q1940655,서울특별시 종로구 북촌로6길 1,W84,37.580051,126.985146,null,null
2.000000,V02Q1940539,서울특별시 종로구 이화장길 33,W84,37.576920,127.004514,null,null
3.000000,V02Q1940737,서울특별시 종로구 평창10길 5,W84,37.605426,126.967435,null,null
4.000000,V02Q1940721,서울특별시 종로구 혜화로 2,W84,37.586059,127.000838,null,null
5.000000,V02Q1940522,서울특별시 종로구 지봉로13길 82,W84,37.577075,127.012367,null,null
6.000000,V02Q1940499,서울특별시 종로구 사직로8길 13,W84,37.574960,126.970272,null,null
7.000000,V02Q1940695,서울특별시 종로구 송월길 154,W84,37.571837,126.962032,null,null
8.000000,V02Q1940716,서울특별시 종로구 통일로14길 36,W84,37.575953,126.958182,null,null
9.000000,V02Q1940649,서울특별시 종로구 경희궁1길 15,W84,37.571340,126.970504,null,null
10.000000,V02Q1940232,서울특별시 종로구 삼청로 107,W84,37.585033,126.981876,null,null


In [20]:
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
No,Double,1170,1170,0,1.000000,1,585.500000,337.894214,1.000000,292.916667,585.500000,878.083333,1170.000000
모델 시리얼(*),String,1170,1170,0,V02Q1940655,1,null,null,OC3CL200010,OC3CL240001,V02Q1940329,V02Q1940624,V02Q2300007
주소,String,1170,1155,0,서울대공원,4,null,null,서울 강동구 강일동 685,서울특별시 강서구 등촌로51가길 26,서울특별시 동작구 상도로 181,서울특별시 송파구 중대로4길 8,서울특별시 중랑구 중랑천로24길 19
좌표 구분코드,String,1170,1,0,W84,1170,null,null,W84,W84,W84,W84,W84
위도,Comparable<*>,1170,1167,0,37.568686,2,null,null,null,null,null,null,null
경도,Comparable<*>,1170,1166,0,127.080180,2,null,null,null,null,null,null,null
변경 전 시리얼,String?,1170,28,1143,OC3CL2000086,1,null,null,OC3CL2000086,OC3DL2200002,OC3DL2200010,OC3DL2200016,OC3KL2400125
변경 전 시리얼(데이터 상 표기),String?,1170,6,1143,OC3DL220001,10,null,null,OC3CL200008,OC3DL220000,OC3DL220001,OC3DL220001,OC3KL240012


In [44]:
USE {
    dependencies("org.xerial:sqlite-jdbc:3.51.1.0")
}

In [45]:
val dbPath = "jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite"
// val dbConfig = DbConnectionConfig(dbPath)

In [19]:
import java.sql.DriverManager
val tableName = "SDoT_Location"
DriverManager.getConnection(dbPath).use{ conn ->

    val sql = """INSERT INTO ${tableName} (serial, addr, lat, lng) VALUES (?,?,?,? )""".trimIndent()
    df.select("모델 시리얼(*)", "주소", "위도", "경도").forEach { it ->
        try {
            conn.prepareStatement(sql)?.use { preparedStatement ->
                preparedStatement.setString(1, it["모델 시리얼(*)"].toString())
                preparedStatement.setString(2, it["주소"].toString())
                preparedStatement.setString(3, it["위도"].toString())
                preparedStatement.setString(4, it["경도"].toString())
                preparedStatement.executeUpdate()
            }
        } catch (e: Exception){
            println(e.localizedMessage)
        }
    }

}

In [31]:
import kotlin.random.Random

val data = df.select("모델 시리얼(*)", "주소", "위도", "경도").add("value") {
    // 소수점 둘레 2자리까지 반올림한 10.0 ~ 100.0 사이의 실수
    String.format("%.2f", Random.nextDouble(10.0, 100.0))
}

data



모델 시리얼(*),주소,위도,경도,value
V02Q1940655,서울특별시 종로구 북촌로6길 1,37.580051,126.985146,78.21
V02Q1940539,서울특별시 종로구 이화장길 33,37.576920,127.004514,66.99
V02Q1940737,서울특별시 종로구 평창10길 5,37.605426,126.967435,83.36
V02Q1940721,서울특별시 종로구 혜화로 2,37.586059,127.000838,34.59
V02Q1940522,서울특별시 종로구 지봉로13길 82,37.577075,127.012367,27.14
V02Q1940499,서울특별시 종로구 사직로8길 13,37.574960,126.970272,32.55
V02Q1940695,서울특별시 종로구 송월길 154,37.571837,126.962032,43.43
V02Q1940716,서울특별시 종로구 통일로14길 36,37.575953,126.958182,31.98
V02Q1940649,서울특별시 종로구 경희궁1길 15,37.571340,126.970504,36.96
V02Q1940232,서울특별시 종로구 삼청로 107,37.585033,126.981876,91.35


In [34]:
val data2 = data.rename(
    "모델 시리얼(*)" to "serial",
    "주소" to "addr",
    "위도" to "lat",
    "경도" to "lng",
    "value" to "value"
)
data2

serial,addr,lat,lng,value
V02Q1940655,서울특별시 종로구 북촌로6길 1,37.580051,126.985146,78.21
V02Q1940539,서울특별시 종로구 이화장길 33,37.576920,127.004514,66.99
V02Q1940737,서울특별시 종로구 평창10길 5,37.605426,126.967435,83.36
V02Q1940721,서울특별시 종로구 혜화로 2,37.586059,127.000838,34.59
V02Q1940522,서울특별시 종로구 지봉로13길 82,37.577075,127.012367,27.14
V02Q1940499,서울특별시 종로구 사직로8길 13,37.574960,126.970272,32.55
V02Q1940695,서울특별시 종로구 송월길 154,37.571837,126.962032,43.43
V02Q1940716,서울특별시 종로구 통일로14길 36,37.575953,126.958182,31.98
V02Q1940649,서울특별시 종로구 경희궁1길 15,37.571340,126.970504,36.96
V02Q1940232,서울특별시 종로구 삼청로 107,37.585033,126.981876,91.35


In [ ]:
val values =  data2.rows().map {
    "{lat:${it.lat}, lng:${it.lng}, value:${it.value}, serial:\"${it.serial}\", addr:\"${it.addr}\"}"
}
values

In [ ]:
val values = data2.map{it}.joinToString(
    separator = ",",
    prefix = "[",
    postfix = "]"
){
    "{lat:${it.lat}, lng:${it.lng}, value:${it.value}, serial:\"${it.serial}\", addr:\"${it.addr}\"}"
}
values


In [ ]:
// 1. 현재 S-DoT 장비의 unique count 값이 1170.
// 2. 최초 1000 건을 수집하되 SENSING_TIME 이 unique 하면 200 건을 더 수집.
// 3. 수집된 데이터중 SENSING_TIME 이 MAX(SENSING_TIME) 인 값만 filtering.

val start_index = 1001
val end_index = 1200

val url = "http://openapi.seoul.go.kr:8088/YourApiKey/json/sDoTEnv/${start_index}/${end_index}/"


//val df = DataFrame.readJson(url)


In [253]:
df.sDoTEnv.row[0].SENSING_TIME.describe()

name,type,count,unique,nulls,top,freq,min,p25,median,p75,max
SENSING_TIME,String,200,1,0,2026-04-02_08:07:00,200,2026-04-02_08:07:00,2026-04-02_08:07:00,2026-04-02_08:07:00,2026-04-02_08:07:00,2026-04-02_08:07:00


In [254]:
df.describe()

name,path,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
list_total_count,"[sDoTEnv, list_total_count]",Int,1,1,0,434921,1,434921.000000,NaN,434921,434921.000000,434921.000000,434921.000000,434921
CODE,"[sDoTEnv, RESULT, CODE]",String,1,1,0,INFO-000,1,null,null,INFO-000,INFO-000,INFO-000,INFO-000,INFO-000
MESSAGE,"[sDoTEnv, RESULT, MESSAGE]",String,1,1,0,정상 처리되었습니다,1,null,null,정상 처리되었습니다,정상 처리되었습니다,정상 처리되었습니다,정상 처리되었습니다,정상 처리되었습니다
MODELNAME,"[sDoTEnv, row, MODELNAME]",String,200,1,0,SDOT001,200,null,null,SDOT001,SDOT001,SDOT001,SDOT001,SDOT001
SERIAL,"[sDoTEnv, row, SERIAL]",String,200,200,0,V02Q1940544,1,null,null,OC3CL200013,V02Q1940272,V02Q1940505,V02Q1940686,V02Q1940952
SENSING_TIME,"[sDoTEnv, row, SENSING_TIME]",String,200,1,0,2026-04-02_08:07:00,200,null,null,2026-04-02_08:07:00,2026-04-02_08:07:00,2026-04-02_08:07:00,2026-04-02_08:07:00,2026-04-02_08:07:00
REGION,"[sDoTEnv, row, REGION]",String,200,7,0,residential_area,132,null,null,commercial_area,residential_area,residential_area,residential_area,traditional_markets
AUTONOMOUS_DISTRICT,"[sDoTEnv, row, AUTONOMOUS_DISTRICT]",String,200,10,0,Gwanak-gu,37,null,null,Dongjak-gu,Geumcheon-gu,Gwanak-gu,Songpa-gu,Yongsan-gu
ADMINISTRATIVE_DISTRICT,"[sDoTEnv, row, ADMINISTRATIVE_DISTRICT]",String,200,131,0,Jong-ro1(il).2(i).3(sam,4,null,null,Amsa1(il)-dong,Gahoe-dong,Jungang-dong,Seokchon-dong,Yongsan2(i)-dong
MAX_TEMP,"[sDoTEnv, row, MAX_TEMP]",String,200,43,0,9.4,15,null,null,,10.9,9.0,9.5,9.9


In [288]:
val url = "http://192.168.35.107:7788/seoul/sdot_env_info"
val df = DataFrame.readJson(url)
df

sensing_time,serial,region,autonomous_district,administrative_district,addr,lat,lng,max_temp,avg_temp,min_temp,max_humi,avg_humi,min_humi,max_ultra_rays,avg_ultra_rays,min_ultra_rays,max_noise,avg_noise,min_noise,max_vibr_x,avg_vibr_x,min_vibr_x,max_vibr_y,avg_vibr_y,min_vibr_y,max_vibr_z,avg_vibr_z,min_vibr_z,max_no2,avg_no2,min_no2,max_co,avg_co,min_co,max_so2,avg_so2,min_so2,max_nh3,avg_nh3,min_nh3,max_h2s,avg_h2s,min_h2s,max_o3,avg_o3,min_o3
2026-04-06_13:07:00,OC3CL200060,residential_area,Gangdong-gu,Dunchon1(il)-dong,서울특별시 강동구 둔촌동 산 29-1,37.522210234,127.145519881,9.9,9.2,9.5,,,,1.4,1.3,1.3,59,55,57,,,,,,,,,,,,,,,,,,,0.0,0.0,0.0,0.0,0.0,0.0,,,
2026-04-06_13:07:00,V02Q1940637,residential_area,Yongsan-gu,Hangang-ro-dong,서울특별시 용산구 서빙고로 35,37.525447033,126.968845768,12.2,11.1,11.7,82.,72.,76.,0.1,0.0,0.1,52,47,50,0.01,0.01,0.01,0.07,0.06,0.07,1.06,1.05,1.05,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940648,residential_area,Yongsan-gu,Itaewon1(il)-dong,서울특별시 용산구 우사단로 25,37.532294579,126.995986461,12.6,11.7,12.3,73.,70.,71.,0.4,0.1,0.2,55,48,52,0.03,0.02,0.02,0.1,0.09,0.09,1.04,1.03,1.04,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940686,residential_area,Yongsan-gu,Itaewon2(i)-dong,서울특별시 용산구 회나무로13나길 21,37.541371627,126.991128742,13.4,12.3,12.8,72.,67.,69.,0.2,0.1,0.1,50,42,44,0.01,0.01,0.01,0.05,0.04,0.04,1.08,1.07,1.07,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940726,residential_area,Yongsan-gu,Yongmun-dong,서울특별시 용산구 새창로12길 11-7,37.539012287,126.957917034,13.1,12.1,12.6,74.,68.,70.,0.2,0.0,0.1,49,45,47,0.01,0.01,0.01,0.07,0.06,0.07,1.05,1.04,1.04,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940602,roads_and_parks,Jung-gu,Hoehyeon-dong,서울측별시 중구 회현동1가 100-115,37.554623618,126.98055898,11.0,9.9,10.4,82.,74.,78.,0.0,0.0,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940608,roads_and_parks,Jung-gu,Myeong-dong,서울특별시 중구 소파로 97,37.55662898,126.985664183,12.1,10.8,11.6,85.,74.,80.,0.2,0.1,0.1,53,49,52,0.07,0.05,0.06,0.05,0.04,0.04,1.05,1.04,1.05,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940780,residential_area,Jung-gu,Sogong-dong,서울특별시 중구 덕수궁길 15,37.564698899,126.975981023,12.3,11.3,12.0,76.,71.,73.,0.4,0.2,0.3,50,49,50,0.05,0.04,0.04,0.05,0.04,0.05,1.05,1.04,1.05,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940572,residential_area,Yongsan-gu,Ichon1(il)-dong,서울측별시 용산구 이촌동 301-156,37.521117972,126.97179047,12.6,11.9,12.2,76.,71.,74.,0.5,0.2,0.3,46,40,43,0.01,0.01,0.01,0.06,0.06,0.06,1.02,1.01,1.01,,,,,,,,,,,,,,,,,,
2026-04-06_13:07:00,V02Q1940567,roads_and_parks,Jung-gu,Eulji-ro-dong,서울특별시 중구 창경궁로5다길 22,37.567955619,126.996509941,11.8,10.4,11.1,81.,73.,76.,0.0,0.0,0.0,53,42,47,0.02,0.01,0.02,0.07,0.06,0.06,1.09,1.07,1.08,,,,,,,,,,,,,,,,,,


In [289]:
df.describe()

name,type,count,unique,nulls,top,freq,min,p25,median,p75,max
sensing_time,String,884,1,0,2026-04-06_13:07:00,884,2026-04-06_13:07:00,2026-04-06_13:07:00,2026-04-06_13:07:00,2026-04-06_13:07:00,2026-04-06_13:07:00
serial,String,884,884,0,OC3CL200060,1,OC3CL200010,OC3CL210300,V02Q1940328,V02Q1940633,V02Q2300007
region,String,884,8,0,residential_area,627,commercial_area,residential_area,residential_area,residential_area,traditional_markets
autonomous_district,String,884,26,0,Gangdong-gu,68,Dobong-gu,Gangnam-gu,Jongno-gu,Seongbuk-gu,Yongsan-gu
administrative_district,String,884,379,0,Cheonho2(i)-dong,11,Ahyeon-dong,Garak2(i)-dong,Majang-dong,Seongsan1(il)-dong,meeting_bridge2
addr,String,884,876,0,서울특별시 종로구 세종로 1-68,3,서울 강동구 강일동 685,서울특별시 강서구 곰달래로19가길 34,서울특별시 동작구 사당로16라길 27,서울특별시 송파구 올림픽로 135,서울특별시 중랑구 중랑천로24길 19
lat,String,884,882,0,37.47129464,2,37.424820304,37.510731657,37.548160591,37.577605,37.690991222
lng,String,884,881,0,126.97655326,2,126.799983213,126.919490435,126.99379344,127.052534714,127.17936066
max_temp,String,884,57,0,12.9,60,,11.9,12.5,13.0,9.9
avg_temp,String,884,55,0,11.4,51,,10.9,11.5,12.1,9.9


In [282]:
val values = df.select("serial", "addr", "lat", "lng", "max_temp", "max_humi", "max_ultra_rays", "max_no2","max_co", "max_so2", "max_nh3", "max_h2s",  "max_o3").map{it}.joinToString(
    separator = ",",
    prefix = "[",
    postfix = "]"
){
    "{lat:${it.lat}, lng:${it.lng}, serial:\"${it.serial}\", addr:\"${it.addr}\" , max_temp:${if(it.max_temp.isEmpty()) 0 else it.max_temp}, max_humi:${if(it.max_humi.isEmpty()) 0 else it.max_humi}, max_ultra_rays:${if(it.max_ultra_rays.isEmpty()) 0 else it.max_ultra_rays}, max_no2:${if(it.max_no2.isEmpty()) 0 else it.max_no2}, max_co:${if(it.max_co.isEmpty()) 0 else it.max_co}, max_so2:${if(it.max_so2.isEmpty()) 0 else it.max_so2}, max_nh3:${if(it.max_nh3.isEmpty()) 0 else it.max_nh3}, max_h2s:${if(it.max_h2s.isEmpty()) 0 else it.max_h2s}, max_o3:${if(it.max_o3.isEmpty()) 0 else it.max_o3} }"
}
File("/Users/unchil/AndroidStudioProjects/OceanWaterInfo/composeApp/src/jvmMain/resources/output2.js").writeText("export const valuesHexagon = " + values + ";")

### 경기도 대기질 측정 정보

In [1]:
%useLatestDescriptors
%use dataframe
%use datetime

In [33]:
val urlStation = "/Users/unchil/Desktop/station_list.json"
val dfStation = DataFrame.readJson(urlStation)
dfStation

obs,addr,op,regdate,lng,lat
가남읍,경기도 여주시 가남읍 태평중앙1길 20 가남읍행정복지센터 옥상,경기도보건환경연구원,2020,127.544876,37.201645
가평,경기도 가평군 가평읍 석봉로 181 가평군청 의회동,경기도보건환경연구원,2010,127.509705,37.831310
감일,경기도 하남시 감이동 469-1 단가람 유치원 인근 지상,경기도보건환경연구원,2025,127.163734,37.506683
경안동,경기 광주시 중앙로 128 농협중앙회,경기도보건환경연구원,2006,127.258018,37.411236
고덕동,경기도 평택시 고덕면 고덕국제2로 111 종덕초등학교 2층 옥상,경기도보건환경연구원,2020,127.046326,37.053226
고색동,경기 수원시 권선구 서부로 1600 수원시도로교통관리사업소,경기도보건환경연구원,2006,126.976418,37.252392
고읍,경기 양주시 고읍남로 205 청소년문화의 집,경기도보건환경연구원,2017,127.084747,37.791687
고잔동,경기 안산시 단원구 화랑로 387 안산시청,경기도보건환경연구원,1987,126.830238,37.322124
고천동,경기 의왕시 시청로 11 의왕시청 민원실,경기도보건환경연구원,1995,126.968803,37.344372
고촌읍,경기 김포시 고촌읍 신곡로 152 김포시상하수도사업소,경기도보건환경연구원,2001,126.763885,37.605469


In [128]:
dfStation.filter { it.obs.contains("안양") }

obs,addr,op,regdate,lng,lat
안양2동,경기 안양시 만안구 안양로 384번길 50 안양2동 행정복지센터,경기도보건환경연구원,2004,126.917824,37.405083
안양8동,경기도 안양시 만안구 문예로36번길 16 안양아트센터 옥상 (안양동),경기도보건환경연구원,1993,126.931358,37.384586


In [ ]:
import java.sql.DriverManager
val tableName = "KDoT_Location"
DriverManager.getConnection(dbPath).use{ conn ->

    val sql = """INSERT INTO ${tableName} (obs, addr, op, regdate, lng, lat) VALUES (?,?,?,?,?,? )""".trimIndent()
    df.forEach { it ->
        try {
            conn.prepareStatement(sql)?.use { preparedStatement ->
                preparedStatement.setString(1, it["obs"].toString())
                preparedStatement.setString(2, it["addr"].toString())
                preparedStatement.setString(3, it["op"].toString())
                preparedStatement.setString(4, it["regdate"].toString())
                preparedStatement.setString(5, it["lng"].toString())
                preparedStatement.setString(6, it["lat"].toString())
                preparedStatement.executeUpdate()
            }
        } catch (e: Exception){
            println(e.localizedMessage)
        }
    }

}

In [ ]:
import java.net.URLEncoder
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern

val now = Clock.System.now()

@OptIn(FormatStringsInDatetimeFormats::class)
val previous1Hour = now
    .minus(1, DateTimeUnit.HOUR)
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH")})

print("Current time : ${now}, Previous time : ${previous1Hour}")


val endPoint = "https://openapi.gg.go.kr"
val service = "Sidoatmospolutnmesure"
val apiKey = "YourApiKey"
val type = "json"

val url = "${endPoint}/${service}?KEY=${apiKey}&Type=${type}&MESURE_DAY_TM=${ URLEncoder.encode("${previous1Hour}:00")}"

url

In [10]:
val url = "https://openapi.gg.go.kr/Sidoatmospolutnmesure?KEY=YourApiKey&Type=json&MESURE_DAY_TM=2026-04-15%2008%3A00"

In [11]:
val df = DataFrame.readJson(url)
df

<iframe onload="o_resize_iframe_out_12()" style="width:100%;" class="result_container" id="iframe_out_12" frameBorder="0" srcdoc=" <html theme='dark'>
 <head>
 <style type="text/css">
 :root {
 --background: #fff;
 --background-odd: #f5f5f5;
 --background-hover: #d9edfd;
 --header-text-color: #474747;
 --text-color: #848484;
 --text-color-dark: #000;
 --text-color-medium: #737373;
 --text-color-pale: #b3b3b3;
 --inner-border-color: #aaa;
 --bold-border-color: #000;
 --link-color: #296eaa;
 --link-color-pale: #296eaa;
 --link-hover: #1a466c;
}

:root[theme="dark"], :root [data-jp-theme-light="false"], .dataframe_dark{
 --background: #303030;
 --background-odd: #3c3c3c;
 --background-hover: #464646;
 --header-text-color: #dddddd;
 --text-color: #b3b3b3;
 --text-color-dark: #dddddd;
 --text-color-medium: #b2b2b2;
 --text-color-pale: #737373;
 --inner-border-color: #707070;
 --bold-border-color: #777777;
 --link-color: #008dc0;
 --link-color-pale: #97e1fb;
 --link-hover: #00688e;
}

p.dataframe_description {
 color: var(--text-color-dark);
}

table.dataframe {
 font-family: "Helvetica Neue", Helvetica, Arial, sans-serif;
 font-size: 12px;
 background-color: var(--background);
 color: var(--text-color-dark);
 border: none;
 border-collapse: collapse;
}

table.dataframe th, td {
 padding: 6px;
 border: 1px solid transparent;
 text-align: left;
}

table.dataframe th {
 background-color: var(--background);
 color: var(--header-text-color);
}

table.dataframe td {
 vertical-align: top;
 white-space: nowrap;
}

table.dataframe th.bottomBorder {
 border-bottom-color: var(--bold-border-color);
}

table.dataframe tbody > tr:nth-child(odd) {
 background: var(--background-odd);
}

table.dataframe tbody > tr:nth-child(even) {
 background: var(--background);
}

table.dataframe tbody > tr:hover {
 background: var(--background-hover);
}

table.dataframe a {
 cursor: pointer;
 color: var(--link-color);
 text-decoration: none;
}

table.dataframe tr:hover > td a {
 color: var(--link-color-pale);
}

table.dataframe a:hover {
 color: var(--link-hover);
 text-decoration: underline;
}

table.dataframe img {
 max-width: fit-content;
}

table.dataframe th.complex {
 background-color: var(--background);
 border: 1px solid var(--background);
}

table.dataframe .leftBorder {
 border-left-color: var(--inner-border-color);
}

table.dataframe .rightBorder {
 border-right-color: var(--inner-border-color);
}

table.dataframe .rightAlign {
 text-align: right;
}

table.dataframe .expanderSvg {
 width: 8px;
 height: 8px;
 margin-right: 3px;
}

table.dataframe .expander {
 display: flex;
 align-items: center;
}

/* formatting */

table.dataframe .null {
 color: var(--text-color-pale);
}

table.dataframe .structural {
 color: var(--text-color-medium);
 font-weight: bold;
}

table.dataframe .dataFrameCaption {
 font-weight: bold;
}

table.dataframe .numbers {
 color: var(--text-color-dark);
}

table.dataframe td:hover .formatted .structural, .null {
 color: var(--text-color-dark);
}

table.dataframe tr:hover .formatted .structural, .null {
 color: var(--text-color-dark);
}


:root {
 --scroll-bg: #f5f5f5;
 --scroll-fg: #b3b3b3;
}
:root[theme="dark"], :root [data-jp-theme-light="false"]{
 --scroll-bg: #3c3c3c;
 --scroll-fg: #97e1fb;
}
body {
 scrollbar-color: var(--scroll-fg) var(--scroll-bg);
}
body::-webkit-scrollbar {
 width: 10px; /* Mostly for vertical scrollbars */
 height: 10px; /* Mostly for horizontal scrollbars */
}
body::-webkit-scrollbar-thumb {
 background-color: var(--scroll-fg);
}
body::-webkit-scrollbar-track {
 background-color: var(--scroll-bg);
}
 </style>
 </head>
 <body>
 <table class="dataframe" id="df_-1140850644"></table>

<p class="dataframe_description">DataFrame: rowsCount = 1, columnsCount = 1</p>

 </body>
 <script>
 (function () {
 window.DataFrame = window.DataFrame || new (function () {
 this.addTable = function (df) {
 let cols = df.cols;
 for (let i = 0; i < cols.length; i++) {
 for (let c of cols[i].children) {
 cols[c].parent = i;
 }

In [76]:
df.Sidoatmospolutnmesure[0].head[0].list_total_count[0]

126

In [78]:
val df_data = df.Sidoatmospolutnmesure[0].row[1]
df_data

SIGUN_CD,SIGUN_NM,MESURSTN_NM,INSTL_YY,MESRNW_NM,MESURE_DAY_TM,SUA_GAS_DNST_VL,COMNXD_DNST_VL,NO2_DNST_VL,OZONE_DNST_VL,FINEDUST_PM10_DNST_VL,FINEDUST_PM2_5_DNST_VL
41190,부천시,소사본동,1986,도시대기,2026-04-10 12:00,0.002000,0.300000,0.014000,0.037000,23,15
41190,부천시,내동,1986,도시대기,2026-04-10 12:00,0.002000,0.300000,0.026000,0.030000,23,24
41190,부천시,중2동,1998,도시대기,2026-04-10 12:00,0.002000,0.300000,0.021000,0.034000,28,8
41190,부천시,오정동,2000,도시대기,2026-04-10 12:00,0.003000,0.600000,0.017000,0.030000,35,24
41190,부천시,송내대로(중동),2004,도로변대기,2026-04-10 12:00,0.002000,0.300000,0.024000,0.029000,33,22
41130,성남시,상대원동,2008,도시대기,2026-04-10 12:00,0.002000,0.500000,0.022000,0.025000,6,3
41150,의정부시,의정부동,1994,도시대기,2026-04-10 12:00,0.003000,0.500000,0.013000,0.038000,9,9
41150,의정부시,의정부1동,2002,도시대기,2026-04-10 12:00,0.004000,0.400000,0.011000,0.035000,10,6
41150,의정부시,송산3동,2020,도시대기,2026-04-10 12:00,0.002000,0.400000,0.030000,0.020000,5,2
41170,안양시,안양8동,1993,도시대기,2026-04-10 12:00,0.003000,0.300000,0.008000,0.044000,11,9


In [12]:
fun loadData(path:String, maxPage:Int): List<DataFrame<*>> {
    val rows = mutableListOf<DataFrame<*>>()
    var requestPage = 1
    do{
        val pagePath = "$path&pIndex=$requestPage"
        val jsonData = DataFrame.readJson(pagePath)
        try {
            val instanceDf = (jsonData["Sidoatmospolutnmesure"][0] as DataFrame<*>)["row"][1] as DataFrame<*>
            requestPage += 1
            rows.add(instanceDf)
        } catch(e: Exception) {
            print(e.localizedMessage)
            break
        }
    } while (requestPage <= maxPage )
    return rows
}


In [13]:
val dfResult = loadData(url, 2).concat()
dfResult

SIGUN_CD,SIGUN_NM,MESURSTN_NM,INSTL_YY,MESRNW_NM,MESURE_DAY_TM,SUA_GAS_DNST_VL,COMNXD_DNST_VL,NO2_DNST_VL,OZONE_DNST_VL,FINEDUST_PM10_DNST_VL,FINEDUST_PM2_5_DNST_VL
41190,부천시,소사본동,1986,도시대기,2026-04-15 08:00,0.004000,0.600000,0.041000,0.025000,96,64
41190,부천시,내동,1986,도시대기,2026-04-15 08:00,0.003000,0.500000,0.070000,0.013000,148,93
41190,부천시,중2동,1998,도시대기,2026-04-15 08:00,0.004000,0.600000,0.044000,0.025000,98,37
41190,부천시,오정동,2000,도시대기,2026-04-15 08:00,0.003000,0.500000,0.041000,0.009000,125,61
41190,부천시,송내대로(중동),2004,도로변대기,2026-04-15 08:00,0.003000,0.500000,0.054000,0.009000,130,77
41270,안산시,중앙대로(고잔동),2009,도로변대기,2026-04-15 08:00,0.004000,0.700000,0.062000,0.008000,90,73
41290,과천시,별양동,1991,도시대기,2026-04-15 08:00,0.003000,0.700000,0.052000,0.012000,68,49
41290,과천시,과천동,2000,도시대기,2026-04-15 08:00,0.003000,0.700000,0.043000,0.018000,56,33
41310,구리시,교문동,1987,도시대기,2026-04-15 08:00,0.003000,0.400000,0.033000,0.021000,51,26
41310,구리시,동구동,2001,도시대기,2026-04-15 08:00,0.004000,0.400000,0.039000,0.005000,71,31


In [14]:
dfResult.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
SIGUN_CD,String,126,31,0,41270,8,null,null,41110,41220,41370,41550,41830
SIGUN_NM,String,126,31,0,안산시,8,null,null,가평군,부천시,안산시,의왕시,화성시
MESURSTN_NM,String,126,126,0,소사본동,1,null,null,가남읍,동구동,소사본동,의정부동,화도읍
INSTL_YY,String,126,34,0,2020,17,null,null,1986,2000,2006,2019,2025
MESRNW_NM,String,126,3,0,도시대기,111,null,null,교외대기,도시대기,도시대기,도시대기,도시대기
MESURE_DAY_TM,String,126,1,0,2026-04-15 08:00,126,null,null,2026-04-15 08:00,2026-04-15 08:00,2026-04-15 08:00,2026-04-15 08:00,2026-04-15 08:00
SUA_GAS_DNST_VL,Float?,126,7,1,0.003000,55,0.003152,0.001063,0.001000,0.002000,0.003000,0.004000,0.006000
COMNXD_DNST_VL,Number?,126,15,1,0.500000,26,0.544000,0.178434,0.200000,0.400000,0.500000,0.700000,1.000000
NO2_DNST_VL,Float?,126,58,1,0.052000,5,0.038168,0.019025,0.003000,0.022000,0.038000,0.053000,0.075000
OZONE_DNST_VL,Float,126,32,0,0.013000,10,0.015873,0.008974,0.004000,0.009000,0.014000,0.022000,0.057000


In [6]:
val dfRename = dfResult.rename(
    "SUA_GAS_DNST_VL" to "SO2",
    "COMNXD_DNST_VL" to "CO",
    "NO2_DNST_VL" to "NO2",
    "OZONE_DNST_VL" to "O3",
    "FINEDUST_PM10_DNST_VL" to "PM10",
    "FINEDUST_PM2_5_DNST_VL" to "PM2.5"
)
dfRename

SIGUN_CD,SIGUN_NM,MESURSTN_NM,INSTL_YY,MESRNW_NM,MESURE_DAY_TM,SO2,CO,NO2,O3,PM10,PM2.5
41190,부천시,소사본동,1986,도시대기,2026-04-15 08:00,0.004000,0.600000,0.041000,0.025000,96,64
41190,부천시,내동,1986,도시대기,2026-04-15 08:00,0.003000,0.500000,0.070000,0.013000,148,93
41190,부천시,중2동,1998,도시대기,2026-04-15 08:00,0.004000,0.600000,0.044000,0.025000,98,37
41190,부천시,오정동,2000,도시대기,2026-04-15 08:00,0.003000,0.500000,0.041000,0.009000,125,61
41190,부천시,송내대로(중동),2004,도로변대기,2026-04-15 08:00,0.003000,0.500000,0.054000,0.009000,130,77
41270,안산시,중앙대로(고잔동),2009,도로변대기,2026-04-15 08:00,0.004000,0.700000,0.062000,0.008000,90,73
41290,과천시,별양동,1991,도시대기,2026-04-15 08:00,0.003000,0.700000,0.052000,0.012000,68,49
41290,과천시,과천동,2000,도시대기,2026-04-15 08:00,0.003000,0.700000,0.043000,0.018000,56,33
41310,구리시,교문동,1987,도시대기,2026-04-15 08:00,0.003000,0.400000,0.033000,0.021000,51,26
41310,구리시,동구동,2001,도시대기,2026-04-15 08:00,0.004000,0.400000,0.039000,0.005000,71,31


In [15]:
dfRename.filter { it.MESURSTN_NM.contains("안양") }

SIGUN_CD,SIGUN_NM,MESURSTN_NM,INSTL_YY,MESRNW_NM,MESURE_DAY_TM,SO2,CO,NO2,O3,PM10,PM2.5
41170,안양시,안양8동,1993,도시대기,2026-04-15 08:00,0.003000,0.700000,0.044000,0.013000,76,59
41170,안양시,안양2동,2004,도시대기,2026-04-15 08:00,0.005000,0.700000,0.058000,0.013000,57,42


In [9]:
dfResult.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
SIGUN_CD,String,126,31,0,41130,8,null,null,41110,41220,41370,41550,41830
SIGUN_NM,String,126,31,0,성남시,8,null,null,가평군,부천시,안산시,의왕시,화성시
MESURSTN_NM,String,126,126,0,소사본동,1,null,null,가남읍,동구동,소사본동,의정부동,화도읍
INSTL_YY,String,126,34,0,2020,17,null,null,1986,2000,2006,2019,2025
MESRNW_NM,String,126,3,0,도시대기,111,null,null,교외대기,도시대기,도시대기,도시대기,도시대기
MESURE_DAY_TM,String,126,1,0,2026-04-11 07:00,126,null,null,2026-04-11 07:00,2026-04-11 07:00,2026-04-11 07:00,2026-04-11 07:00,2026-04-11 07:00
SUA_GAS_DNST_VL,Float?,126,6,1,0.002000,57,0.002336,0.000832,0.001000,0.002000,0.002000,0.003000,0.005000
COMNXD_DNST_VL,Float?,126,6,1,0.500000,58,0.517600,0.080386,0.400000,0.500000,0.500000,0.600000,0.800000
NO2_DNST_VL,Float,126,20,0,0.009000,23,0.009825,0.004252,0.003000,0.007000,0.009000,0.011000,0.032000
OZONE_DNST_VL,Float,126,24,0,0.048000,16,0.046516,0.005306,0.026000,0.043917,0.047000,0.050000,0.061000


In [11]:
val renameDf = dfStation.rename("obs" to "MESURSTN_NM")
renameDf

MESURSTN_NM,addr,op,regdate,lng,lat
가남읍,경기도 여주시 가남읍 태평중앙1길 20 가남읍행정복지센터 옥상,경기도보건환경연구원,2020,127.544876,37.201645
가평,경기도 가평군 가평읍 석봉로 181 가평군청 의회동,경기도보건환경연구원,2010,127.509705,37.831310
감일,경기도 하남시 감이동 469-1 단가람 유치원 인근 지상,경기도보건환경연구원,2025,127.163734,37.506683
경안동,경기 광주시 중앙로 128 농협중앙회,경기도보건환경연구원,2006,127.258018,37.411236
고덕동,경기도 평택시 고덕면 고덕국제2로 111 종덕초등학교 2층 옥상,경기도보건환경연구원,2020,127.046326,37.053226
고색동,경기 수원시 권선구 서부로 1600 수원시도로교통관리사업소,경기도보건환경연구원,2006,126.976418,37.252392
고읍,경기 양주시 고읍남로 205 청소년문화의 집,경기도보건환경연구원,2017,127.084747,37.791687
고잔동,경기 안산시 단원구 화랑로 387 안산시청,경기도보건환경연구원,1987,126.830238,37.322124
고천동,경기 의왕시 시청로 11 의왕시청 민원실,경기도보건환경연구원,1995,126.968803,37.344372
고촌읍,경기 김포시 고촌읍 신곡로 152 김포시상하수도사업소,경기도보건환경연구원,2001,126.763885,37.605469


In [12]:
val df_data = dfResult.leftJoin(renameDf){
    it["MESURSTN_NM"]
}
df_data

SIGUN_CD,SIGUN_NM,MESURSTN_NM,INSTL_YY,MESRNW_NM,MESURE_DAY_TM,SUA_GAS_DNST_VL,COMNXD_DNST_VL,NO2_DNST_VL,OZONE_DNST_VL,FINEDUST_PM10_DNST_VL,FINEDUST_PM2_5_DNST_VL,addr,op,regdate,lng,lat
41190,부천시,소사본동,1986,도시대기,2026-04-11 07:00,0.002000,0.400000,0.009000,0.050000,null,37,경기 부천시 경인옛로 73 소사어울마당 소향관,한국환경공단 수도권동부환경본부,1986,126.799942,37.480042
41190,부천시,내동,1986,도시대기,2026-04-11 07:00,0.002000,0.400000,0.014000,0.046000,43,39,경기 부천시 삼작로 109 신흥동주민센터 앞 도로변,경기도보건환경연구원,1986,126.773331,37.520458
41190,부천시,중2동,1998,도시대기,2026-04-11 07:00,0.002000,0.400000,0.013000,0.050000,44,14,경기 부천시 심중로 121 책마루도서관,경기도보건환경연구원,1998,126.770096,37.493965
41190,부천시,오정동,2000,도시대기,2026-04-11 07:00,0.003000,0.400000,0.013000,0.043000,43,38,경기 부천시 성오로 172 오정어울마당 아트홀,경기도보건환경연구원,2000,126.796021,37.528347
41190,부천시,송내대로(중동),2004,도로변대기,2026-04-11 07:00,0.002000,0.400000,0.017000,0.046000,51,37,null,null,null,null,null
41130,성남시,상대원동,2008,도시대기,2026-04-11 07:00,0.002000,0.600000,0.017000,0.040000,40,24,경기 성남시 중원구 둔촌대로 425 상대원1동행정복지센터,경기도보건환경연구원,2008,127.164383,37.433208
41150,의정부시,의정부동,1994,도시대기,2026-04-11 07:00,0.003000,0.500000,0.009000,0.048000,30,25,경기 의정부시 범골로 138 경기도도로사업소,경기도보건환경연구원,1994,127.040810,37.735561
41150,의정부시,의정부1동,2002,도시대기,2026-04-11 07:00,0.003000,0.500000,0.008000,0.046000,31,27,경기 의정부시 가능로152번길 14 의정부1동 작은도서관,경기도보건환경연구원,2002,127.047607,37.746441
41150,의정부시,송산3동,2020,도시대기,2026-04-11 07:00,0.003000,0.500000,0.015000,0.042000,23,15,경기도 의정부시 민락로243번길 94 푸른마당 근린공원 인근 연결녹...,경기도보건환경연구원,2020,127.106621,37.747669
41170,안양시,안양8동,1993,도시대기,2026-04-11 07:00,0.003000,0.500000,0.007000,0.048000,40,39,경기도 안양시 만안구 문예로36번길 16 안양아트센터 옥상 (안양동),경기도보건환경연구원,1993,126.931358,37.384586


In [13]:
df_data.describe()


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
SIGUN_CD,String,126,31,0,41130,8,null,null,41110,41220,41370,41550,41830
SIGUN_NM,String,126,31,0,성남시,8,null,null,가평군,부천시,안산시,의왕시,화성시
MESURSTN_NM,String,126,126,0,소사본동,1,null,null,가남읍,동구동,소사본동,의정부동,화도읍
INSTL_YY,String,126,34,0,2020,17,null,null,1986,2000,2006,2019,2025
MESRNW_NM,String,126,3,0,도시대기,111,null,null,교외대기,도시대기,도시대기,도시대기,도시대기
MESURE_DAY_TM,String,126,1,0,2026-04-11 07:00,126,null,null,2026-04-11 07:00,2026-04-11 07:00,2026-04-11 07:00,2026-04-11 07:00,2026-04-11 07:00
SUA_GAS_DNST_VL,Float?,126,6,1,0.002000,57,0.002336,0.000832,0.001000,0.002000,0.002000,0.003000,0.005000
COMNXD_DNST_VL,Float?,126,6,1,0.500000,58,0.517600,0.080386,0.400000,0.500000,0.500000,0.600000,0.800000
NO2_DNST_VL,Float,126,20,0,0.009000,23,0.009825,0.004252,0.003000,0.007000,0.009000,0.011000,0.032000
OZONE_DNST_VL,Float,126,24,0,0.048000,16,0.046516,0.005306,0.026000,0.043917,0.047000,0.050000,0.061000


In [14]:
df_data.filter { it.lng == null  }


SIGUN_CD,SIGUN_NM,MESURSTN_NM,INSTL_YY,MESRNW_NM,MESURE_DAY_TM,SUA_GAS_DNST_VL,COMNXD_DNST_VL,NO2_DNST_VL,OZONE_DNST_VL,FINEDUST_PM10_DNST_VL,FINEDUST_PM2_5_DNST_VL,addr,op,regdate,lng,lat
41190,부천시,송내대로(중동),2004,도로변대기,2026-04-11 07:00,0.002000,0.400000,0.017000,0.046000,51,37,null,null,null,null,null
41270,안산시,중앙대로(고잔동),2009,도로변대기,2026-04-11 07:00,0.002000,0.500000,0.019000,0.034000,34,35,null,null,null,null,null
41390,시흥시,서해안로,2020,도로변대기,2026-04-11 07:00,0.003000,0.500000,0.011000,0.056000,38,35,null,null,null,null,null
41360,남양주시,경춘로,2019,도로변대기,2026-04-11 07:00,0.003000,0.500000,0.017000,0.036000,40,40,null,null,null,null,null
41480,파주시,파주,2017,교외대기,2026-04-11 07:00,0.001000,0.500000,0.004000,0.041000,23,20,null,null,null,null,null
41280,고양시,백마로(마두역),2004,도로변대기,2026-04-11 07:00,0.003000,0.400000,0.007000,0.048000,43,31,null,null,null,null,null
41460,용인시,중부대로(구갈동),2011,도로변대기,2026-04-11 07:00,0.002000,0.600000,0.018000,0.043000,44,33,null,null,null,null,null
41500,이천시,설성면,2000,교외대기,2026-04-11 07:00,0.002000,0.500000,0.006000,0.042000,37,38,null,null,null,null,null
41650,포천시,관인면,2001,교외대기,2026-04-11 07:00,0.001000,0.600000,0.005000,0.048000,21,21,null,null,null,null,null
41570,김포시,한강로,2020,도로변대기,2026-04-11 07:00,0.003000,0.400000,0.005000,0.053000,39,30,null,null,null,null,null


In [43]:
import java.nio.charset.StandardCharsets

URLEncoder.encode("2026-04-13 19:00", StandardCharsets.UTF_8.toString())
.replace("+", "%20")

2026-04-13%2019%3A00